In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import json as json_lib
from shapely.geometry import Point, Polygon
from itertools import product
from pathlib import Path
from scipy.special import gamma
import warnings
import time

from py_wake.site import XRSite
from py_wake.wind_turbines import WindTurbines
from py_wake.wind_turbines.generic_wind_turbines import GenericWindTurbine
from py_wake.deficit_models import TurboGaussianDeficit
from py_wake.superposition_models import SquaredSum
from py_wake.wind_farm_models import All2AllIterative

/rds/general/user/vk825/home/envs/wake_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_DIR = Path('../data')
ERA5_DIR = DATA_DIR / 'era5'
OUTPUT_DIR = DATA_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Reproducibility
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

#### Load Norther's polygon from your processed data.

In [3]:
import geopandas as gpd
candidate_farms = gpd.read_file(DATA_DIR / 'candidate_farms.gpkg')
target_gdf = candidate_farms.loc[candidate_farms['name'] == 'Norther'].copy()
if target_gdf.crs.is_geographic:
    target_gdf = target_gdf.to_crs(epsg=32631)
target_poly = target_gdf.geometry.iloc[0]

target_centroid = target_poly.centroid
print(f'Target polygon area: {target_poly.area/1e6:.2f} km\u00b2')
print(f'Target centroid: ({target_centroid.x:.0f}, {target_centroid.y:.0f})')

Target polygon area: 38.41 km²
Target centroid: (500912, 5708426)


##### Load ERA5 data

In [5]:
ds = xr.open_mfdataset(str(ERA5_DIR / 'era5_norther_100m_*.nc'))

In [6]:
# Average across ERA5 grid points around the target site
spatial_dims = [d for d in ['latitude', 'longitude'] if d in ds['u100'].dims]
u100 = ds['u100'].mean(dim=spatial_dims).values
v100 = ds['v100'].mean(dim=spatial_dims).values

# Wind speed
ws_era5 = np.sqrt(u100**2 + v100**2)

# Meteorological wind direction: direction wind is COMING FROM.
# 0 deg = from North, 90 deg = from East.
wd_era5 = (270 - np.degrees(np.arctan2(v100, u100))) % 360

print(f'Mean wind speed: {ws_era5.mean():.2f} m/s')
print(f'Min/Max wind speed: {ws_era5.min():.2f} / {ws_era5.max():.2f} m/s')

Mean wind speed: 8.80 m/s
Min/Max wind speed: 0.00 / 27.65 m/s


In [8]:
from scipy.stats import weibull_min

wd_centres = np.arange(0, 360, 30)
wind_rose_rows = []

for wd in wd_centres:
    lower = (wd - 15) % 360
    upper = (wd + 15) % 360

    if lower > upper:
        mask = (wd_era5 >= lower) | (wd_era5 < upper)
    else:
        mask = (wd_era5 >= lower) & (wd_era5 < upper)

    speeds = ws_era5[mask]
    speeds = speeds[np.isfinite(speeds) & (speeds > 0)]

    frequency = len(speeds) / len(ws_era5)

    if len(speeds) < 5:
        A, k = np.nan, np.nan
    else:
        k, loc, A = weibull_min.fit(speeds, floc=0)

    wind_rose_rows.append({
        'wd': wd,
        'sector_frequency': frequency,
        'weibull_A': A,
        'weibull_k': k,
        'n_samples': len(speeds),
    })

wind_rose = pd.DataFrame(wind_rose_rows)

sector_freq = wind_rose["sector_frequency"].values
weibull_A = wind_rose["weibull_A"].values
weibull_k = wind_rose["weibull_k"].values
mean_ws_per_sector = wind_rose["weibull_A"]* gamma(1 + 1/wind_rose["weibull_k"])
overall_mean_ws = float(np.sum(sector_freq * mean_ws_per_sector))
dominant_wd = float(wd_centres[np.argmax(sector_freq)])

print(wind_rose)
print('Frequency sum:', wind_rose['sector_frequency'].sum())


     wd  sector_frequency  weibull_A  weibull_k  n_samples
0     0          0.056037   8.444577   2.267462       1474
1    30          0.081851   8.858454   2.784762       2153
2    60          0.071244   8.853347   2.597216       1874
3    90          0.056037   8.141152   2.515084       1474
4   120          0.053224   7.208759   2.369640       1400
5   150          0.048054   7.768672   2.140386       1264
6   180          0.090062  10.526288   2.297022       2369
7   210          0.128574  11.510080   2.392055       3382
8   240          0.197004  11.895142   2.707229       5182
9   270          0.099034  10.311149   2.142304       2605
10  300          0.065845   9.489744   2.042344       1732
11  330          0.053034   8.522368   2.015897       1395
Frequency sum: 1.0


#### Creating Site and defining turbines

In [9]:
def create_site(ti=0.06):
    """Create XRSite from wind rose with specified TI."""
    return XRSite(
        ds=xr.Dataset(
            data_vars={
                'Sector_frequency': ('wd', sector_freq),
                'Weibull_A': ('wd', weibull_A),
                'Weibull_k': ('wd', weibull_k),
                'TI': ti,
            },
            coords={'wd': wd_centres}
        )
    )

In [10]:
def create_multi_type_wake_model(target_spec, neighbour_spec, ti=0.06):
    """
    Create PyWake model with TWO turbine types:
      Type 0 = Target (V164-8.0MW)
      Type 1 = Neighbour (varies)
    
    This ensures target turbines are always simulated with correct power curve
    regardless of what neighbour turbine is used.
    """
    site = create_site(ti=ti)
    
    target_wt = GenericWindTurbine(
        name=target_spec['name'],
        diameter=target_spec['diameter'],
        hub_height=target_spec['hub_height'],
        power_norm=target_spec['power_kw'],
        turbulence_intensity=ti,
    )
    
    neighbour_wt = GenericWindTurbine(
        name=neighbour_spec['name'],
        diameter=neighbour_spec['diameter'],
        hub_height=neighbour_spec['hub_height'],
        power_norm=neighbour_spec['power_kw'],
        turbulence_intensity=ti,
    )
    
    # Combine into multi-type WindTurbines object
    wt_combined = WindTurbines.from_WindTurbine_lst([target_wt, neighbour_wt])
    
    return All2AllIterative(
        site, wt_combined,
        wake_deficitModel=TurboGaussianDeficit(),
        superpositionModel=SquaredSum(),
    )

#### Layout generation

In [11]:
def place_turbines_in_polygon(polygon, n_turbines, min_spacing_D=5, rotor_diameter=164, seed=42):
    """
    Place turbines inside a polygon on a regular grid, then randomly select n_turbines.
    Deterministic for a given seed.
    """
    if n_turbines <= 0:
        return np.array([]), np.array([])

    rng = np.random.default_rng(seed)
    min_spacing_m = min_spacing_D * rotor_diameter
    minx, miny, maxx, maxy = polygon.bounds

    # Estimate spacing to spread turbines across polygon
    estimated_spacing = np.sqrt(polygon.area / n_turbines) * 0.9
    spacing = max(estimated_spacing, min_spacing_m)

    x_values = np.arange(minx + spacing / 2, maxx, spacing)
    y_values = np.arange(miny + spacing / 2, maxy, spacing)

    candidates = []
    for x in x_values:
        for y in y_values:
            if polygon.contains(Point(x, y)):
                candidates.append((x, y))

    if len(candidates) == 0:
        return np.array([]), np.array([])

    if len(candidates) < n_turbines:
        selected = np.array(candidates)
    else:
        idx = rng.choice(len(candidates), size=n_turbines, replace=False)
        selected = np.array(candidates)[idx]

    return selected[:, 0], selected[:, 1]

In [12]:
def create_synthetic_neighbour(target_centroid, distance_km, direction_deg, width_km=5, depth_km=5):
    """
    Create a rectangular polygon at a given distance and bearing from target centroid.
    direction_deg: bearing (0=N, 90=E, 180=S, 270=W).
    """
    dist_m = distance_km * 1000
    theta = np.deg2rad(direction_deg)

    center_x = target_centroid.x + dist_m * np.sin(theta)
    center_y = target_centroid.y + dist_m * np.cos(theta)

    half_w = width_km * 1000 / 2
    half_d = depth_km * 1000 / 2

    return Polygon([
        (center_x - half_w, center_y - half_d),
        (center_x + half_w, center_y - half_d),
        (center_x + half_w, center_y + half_d),
        (center_x - half_w, center_y + half_d),
    ])


In [13]:
def target_neighbor_angle(dx, dy):
    '''Returns the angle between the target farm and the neighbour farm'''
    return np.degrees(np.arctan2(dx, dy)) %360

In [14]:
def flow_unit_vector_from_wd(wd_deg):
    """Convert met wind direction (FROM) to flow unit vector (TO)."""
    to_dir = (wd_deg + 180) % 360
    theta = np.deg2rad(to_dir)
    return np.sin(theta), np.cos(theta)

In [15]:
def get_wind_features(angle):
    """
    Get wind properties from the sector closest to the neighbour bearing.
    This tells us: how strong/frequent is the wind blowing FROM the neighbour toward target?
    """
    # Find closest sector
    diffs = np.abs(wd_centres - angle)
    diffs = np.minimum(diffs, 360 - diffs)
    idx = np.argmin(diffs)
    return {
        'wind_speed_from_neighbour_to_target_dir': float(mean_ws_per_sector[idx]),
        'wind_freq_from_neighbour_to_target_dir': float(sector_freq[idx]),
    }


In [16]:
def compute_static_features(target_poly, nb_poly, nb_n_turbines, nb_spacing_D, nb_spec):
    """Compute features that do NOT depend on wind direction."""
    target_centroid = target_poly.centroid
    neighbour_centroid = nb_poly.centroid
    dx = neighbour_centroid.x - target_centroid.x
    dy = neighbour_centroid.y - target_centroid.y
    distance_km = np.sqrt(dx**2 + dy**2) / 1000
    angle = target_neighbor_angle(dx, dy)
    radians = np.deg2rad(angle)
    area_sqkm = nb_poly.area / 1e6
    capacity_mw = nb_n_turbines * nb_spec['power_kw'] / 1000

    wind_feats = get_wind_features(angle)

    return {
        'nb_distance_km': distance_km,
        'nb_direction_deg': angle,
        'nb_dir_sin': np.sin(radians),
        'nb_dir_cos': np.cos(radians),
        'nb_area_sqkm': area_sqkm,
        'nb_n_turbines': nb_n_turbines,
        'nb_spacing_D': nb_spacing_D,
        'nb_capacity_mw': capacity_mw,
        'nb_rotor_diam_m': nb_spec['diameter'],
        'nb_hub_height_m': nb_spec['hub_height'],
        'nb_rated_power_kw': nb_spec['power_kw'],
        'wind_speed_from_neighbour_to_target_dir': wind_feats['wind_speed_from_neighbour_to_target_dir'],
        'wind_freq_from_neighbour_to_target_dir': wind_feats['wind_freq_from_neighbour_to_target_dir'],
        'dominant_wind_dir': dominant_wd,
        'mean_wind_speed': overall_mean_ws,
    }

In [17]:
def compute_wind_relative_features(target_poly, nb_poly, wd_deg):
    """Compute streamwise and lateral distance relative to wind direction."""
    target_centroid = target_poly.centroid
    neighbour_centroid = nb_poly.centroid
    dx = neighbour_centroid.x - target_centroid.x
    dy = neighbour_centroid.y - target_centroid.y

    fx, fy = flow_unit_vector_from_wd(wd_deg)

    # Positive streamwise = neighbour is UPWIND of target
    streamwise_m = -(dx * fx + dy * fy)
    lateral_m = dx * fy - dy * fx

    return {
        'streamwise_distance_km': streamwise_m / 1000,
        'lateral_distance_km': lateral_m / 1000,
        'abs_lateral_distance_km': abs(lateral_m) / 1000,
        'is_upwind': int(streamwise_m > 0),
    }

In [18]:
target_turbine_spec = {
    'name': 'Vestas_V164_8p4MW',
    'diameter': 164,
    'hub_height': 105,
    'power_kw': 8400,
}
TARGET_N_TURBINES = 44
TARGET_SPACING_D = 5

# NEIGHBOURS — VARIED (5 real Belgian offshore specs)
neighbour_turbine_specs = [
    {'name': 'Vestas_V164_8p4MW',  'diameter': 164, 'hub_height': 105, 'power_kw': 8400},  # Norther-like
    {'name': 'SWT_7p0_154',        'diameter': 154, 'hub_height': 106, 'power_kw': 7000},  # Rentel
    {'name': 'SG_8p0_167DD',       'diameter': 167, 'hub_height': 107, 'power_kw': 8000},  # Seamade/Mermaid
    {'name': 'Vestas_V112_3p3MW',  'diameter': 112, 'hub_height': 72,  'power_kw': 3300},  # Nobelwind
    {'name': 'Senvion_6p2M152',    'diameter': 152, 'hub_height': 94,  'power_kw': 6150},  # C-Power
]

# Generate target layout ONCE
x_target, y_target = place_turbines_in_polygon(
    target_poly,
    n_turbines=TARGET_N_TURBINES,
    min_spacing_D=TARGET_SPACING_D,
    rotor_diameter=target_turbine_spec['diameter'],
    seed=RANDOM_SEED,
)


In [19]:
def simulate_aep_scenario(scenario_id, nb_poly, nb_n_turbines, nb_spacing_D, nb_turbine_spec, ti, layout_seed):
    """
    AEP-level simulation: returns ONE row with annual wake loss %.
    Uses the full Weibull wind rose internally.
    """
    # Create multi-type wake model
    wfm = create_multi_type_wake_model(target_turbine_spec, nb_turbine_spec, ti=ti)

    # Place neighbour turbines
    x_nb, y_nb = place_turbines_in_polygon(
        nb_poly, n_turbines=nb_n_turbines, min_spacing_D=nb_spacing_D,
        rotor_diameter=nb_turbine_spec['diameter'], seed=layout_seed
    )
    if len(x_nb) == 0:
        return None

    # Combine layouts
    all_x = np.concatenate([x_target, x_nb])
    all_y = np.concatenate([y_target, y_nb])
    n_target = len(x_target)
    types = np.array([0]*n_target + [1]*len(x_nb))  # 0=target, 1=neighbour

    ws_range = np.arange(3, 26, 1)

    # AEP: target alone
    sim_alone = wfm(x_target, y_target, type=[0]*n_target, wd=wd_centres, ws=ws_range)
    # NOTE: PyWake aep() returns values in GWh. Verify magnitude on first run.
    aep_alone = float(sim_alone.aep().sum().values)

    # AEP: target + neighbour
    sim_with = wfm(all_x, all_y, type=types, wd=wd_centres, ws=ws_range)
    aep_with = float(sim_with.aep().isel(wt=slice(0, n_target)).sum().values)

    # Wake loss
    wake_loss_pct = (1 - aep_with / aep_alone) * 100 if aep_alone > 0 else np.nan

    # Features
    static_feats = compute_static_features(target_poly, nb_poly, len(x_nb), nb_spacing_D, nb_turbine_spec)

    return {
        'scenario_id': scenario_id,
        'ti': ti,
        'nb_turbine_name': nb_turbine_spec['name'],
        'target_n_turbines': n_target,
        'target_spacing_D': TARGET_SPACING_D,
        **static_feats,
        'aep_alone_gwh': aep_alone,
        'aep_with_neighbour_gwh': aep_with,
        'wake_loss_pct': wake_loss_pct,
    }

In [20]:
nb_distances_km = [5, 10, 15, 20, 25, 30]
nb_bearings_deg = np.arange(0, 360, 30)
nb_sizes = [20, 40, 60, 80]
nb_spacings_D = [5, 7, 9]
ti_values = [0.04, 0.06, 0.08, 0.10]  # Use [0.06] for quick test

# Build scenario list
synthetic_scenarios = []
scenario_id = 0

for dist_km, bearing_deg, nb_n, nb_spacing, ti, nb_spec in product(
    nb_distances_km, nb_bearings_deg, nb_sizes, nb_spacings_D, ti_values, neighbour_turbine_specs
):
    scenario_id += 1
    nb_poly = create_synthetic_neighbour(
        target_centroid,
        distance_km=dist_km,
        direction_deg=bearing_deg,
        width_km=5,
        depth_km=5,
    )
    synthetic_scenarios.append({
        'scenario_id': scenario_id,
        'nb_poly': nb_poly,
        'nb_n_turbines': nb_n,
        'nb_spacing_D': nb_spacing,
        'nb_turbine_spec': nb_spec,
        'ti': ti,
        'layout_seed': RANDOM_SEED + scenario_id,
    })

print(f'Total synthetic scenarios: {len(synthetic_scenarios):,}')


Total synthetic scenarios: 17,280


In [21]:
SCENARIOS_TO_RUN = synthetic_scenarios.copy()
SAVE_INTERVAL = 50  

# ============================================================
# SIMULATION LOOP
# ============================================================

aep_rows = []
t_start = time.time()
    
for i, s in enumerate(SCENARIOS_TO_RUN, start=1):
    try:
        row = simulate_aep_scenario(
            scenario_id=s['scenario_id'],
            nb_poly=s['nb_poly'],
            nb_n_turbines=s['nb_n_turbines'],
            nb_spacing_D=s['nb_spacing_D'],
            nb_turbine_spec=s['nb_turbine_spec'],
            ti=s['ti'],
            layout_seed=s['layout_seed'],
            )
        if row is not None:
            aep_rows.append(row)
    except Exception as e:
         print(f'  [FAILED] Scenario {s["scenario_id"]}: {e}')

        # Progress + crash-safe save
    if i % SAVE_INTERVAL == 0:
        elapsed = time.time() - t_start
        rate = i / elapsed * 60  # scenarios per minute
        print(f'  [{i}/{len(SCENARIOS_TO_RUN)}] {len(aep_rows)} rows | {rate:.1f} scen/min')
        pd.DataFrame(aep_rows).to_csv(OUTPUT_DIR / 'training_data_aep_partial.csv', index=False)

# Final save
df_aep = pd.DataFrame(aep_rows)
aep_path = OUTPUT_DIR / 'training_data_aep_final.csv'
df_aep.to_csv(aep_path, index=False)
    
elapsed = time.time() - t_start
print(f'\n=== AEP Simulation Complete ===')
print(f'Rows: {len(df_aep):,}')
print(f'Time: {elapsed/60:.1f} minutes')
print(f'Saved: {aep_path}')
display(df_aep.head())



=== AEP Simulation Complete ===
Rows: 20
Time: 0.1 minutes
Saved: ../data/training_data_aep_final.csv


,scenario_id,ti,nb_turbine_name,target_n_turbines,target_spacing_D,nb_distance_km,nb_direction_deg,nb_dir_sin,nb_dir_cos,nb_area_sqkm,...,nb_rotor_diam_m,nb_hub_height_m,nb_rated_power_kw,wind_speed_from_neighbour_to_target_dir,wind_freq_from_neighbour_to_target_dir,dominant_wind_dir,mean_wind_speed,aep_alone_gwh,aep_with_neighbour_gwh,wake_loss_pct
0,1,0.04,Vestas_V164_8p4MW,44,5,5.0,6.670107e-13,1.164153e-14,1.0,25.0,...,164,105,8400,7.480091,0.056037,240.0,8.79462,1175.460061,1132.901292,3.620605
1,2,0.04,SWT_7p0_154,44,5,5.0,6.670107e-13,1.164153e-14,1.0,25.0,...,154,106,7000,7.480091,0.056037,240.0,8.79462,1175.460061,1141.165808,2.917517
2,3,0.04,SG_8p0_167DD,44,5,5.0,6.670107e-13,1.164153e-14,1.0,25.0,...,167,107,8000,7.480091,0.056037,240.0,8.79462,1175.460061,1132.963358,3.615325
3,4,0.04,Vestas_V112_3p3MW,44,5,5.0,6.670107e-13,1.164153e-14,1.0,25.0,...,112,72,3300,7.480091,0.056037,240.0,8.79462,1175.460061,1154.389223,1.792561
4,5,0.04,Senvion_6p2M152,44,5,5.0,6.670107e-13,1.164153e-14,1.0,25.0,...,152,94,6150,7.480091,0.056037,240.0,8.79462,1175.460061,1144.660932,2.620177
